# Employee Attrition Prediction - ML Model Training

## Objective
Train and compare multiple machine learning models to predict employee attrition using the engineered features from `gold.gold_ml_features`.

## Models Trained
1. **Random Forest** - Best for feature importance and handling non-linear relationships
2. **Logistic Regression** - Interpretable, good baseline
3. **Gradient Boosted Trees** - High accuracy, ensemble learning

## MLflow Integration
- Track all experiments with hyperparameters and metrics
- Register best model to MLflow Model Registry
- Enable model versioning and deployment

## Output
- Trained models logged to MLflow
- Predictions saved to `gold.gold_attrition_predictions`
- Feature importance analysis
- Model performance comparison

---
**Created:** February 5, 2026  
**Author:** Data Engineering Team

## 1. Environment Setup & Imports

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# PySpark ML libraries
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# MLflow
import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ All libraries imported successfully")
print(f"MLflow version: {mlflow.__version__}")
print(f"Execution time: {datetime.now()}")

## 2. Load ML Features from Gold Layer

In [ ]:
# Load the gold ML features table
print("Loading ML features from gold.gold_ml_features...")

ml_features_df = spark.sql("""
    SELECT * 
    FROM workspace.gold.gold_ml_features
""")

# Display basic info
print(f"Total records: {ml_features_df.count():,}")
print(f"Total features: {len(ml_features_df.columns)}")
print("\nSchema:")
ml_features_df.printSchema()

# Display first few rows
display(ml_features_df.limit(5))

## 3. Data Preparation & Feature Engineering

In [ ]:
# Define feature columns (exclude target, ID, and metadata columns)
exclude_cols = ['EmpID', 'is_active', 'features_created_at', 'last_updated']

feature_columns = [col for col in ml_features_df.columns if col not in exclude_cols]

print(f"Total feature columns: {len(feature_columns)}")
print("\nFeature columns:")
for i, col in enumerate(feature_columns, 1):
    print(f"{i}. {col}")

In [ ]:
# Check target variable distribution
print("Target Variable Distribution (is_active):")
target_dist = ml_features_df.groupBy('is_active').count().orderBy('is_active')
display(target_dist)

# Calculate class balance
total_count = ml_features_df.count()
active_count = ml_features_df.filter(F.col('is_active') == 1).count()
attrited_count = ml_features_df.filter(F.col('is_active') == 0).count()

print(f"\nClass Balance:")
print(f"Active employees (0 - No Attrition): {active_count:,} ({active_count/total_count*100:.2f}%)")
print(f"Attrited employees (1 - Attrition): {attrited_count:,} ({attrited_count/total_count*100:.2f}%)")
print(f"\nClass imbalance ratio: {max(active_count, attrited_count) / min(active_count, attrited_count):.2f}:1")

In [ ]:
# Create label column (1 = attrited, 0 = active)
# NOTE: We invert is_active to make 1 = attrition for better ML interpretation
ml_data = ml_features_df.withColumn(
    'label',
    F.when(F.col('is_active') == 0, 1).otherwise(0)  # 1 = attrited, 0 = retained
)

print("✅ Label column created (1 = Attrition, 0 = Retained)")
print("\nLabel distribution:")
display(ml_data.groupBy('label').count().orderBy('label'))

In [ ]:
# Assemble features into a single vector column
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features_raw",
    handleInvalid="skip"  # Skip rows with invalid values
)

# Scale features for models that benefit from it (Logistic Regression)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

print("✅ Feature assembler and scaler configured")

## 4. Train/Test Split

In [ ]:
# Split data into training and testing sets (80/20 split)
train_df, test_df = ml_data.randomSplit([0.8, 0.2], seed=42)

print(f"Training set size: {train_df.count():,} records")
print(f"Test set size: {test_df.count():,} records")
print(f"Total: {ml_data.count():,} records")

# Check class distribution in train/test
print("\nTraining set distribution:")
display(train_df.groupBy('label').count())

print("\nTest set distribution:")
display(test_df.groupBy('label').count())

## 5. Model Training with MLflow

### 5.1 Random Forest Classifier

In [ ]:
# Set MLflow experiment
mlflow.set_experiment("/employee-attrition-prediction")

print("✅ MLflow experiment set: /employee-attrition-prediction")

In [ ]:
# Train Random Forest Model
with mlflow.start_run(run_name="Random_Forest_v1") as run:
    
    print("🌲 Training Random Forest Classifier...")
    
    # Log parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("num_trees", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("train_test_split", "80/20")
    mlflow.log_param("random_seed", 42)
    mlflow.log_param("feature_count", len(feature_columns))
    
    # Define Random Forest model
    rf_classifier = RandomForestClassifier(
        labelCol="label",
        featuresCol="features",
        numTrees=100,
        maxDepth=10,
        maxBins=32,
        seed=42,
        featureSubsetStrategy="auto"
    )
    
    # Create pipeline
    rf_pipeline = Pipeline(stages=[assembler, scaler, rf_classifier])
    
    # Train model
    rf_model = rf_pipeline.fit(train_df)
    
    # Make predictions on test set
    rf_predictions = rf_model.transform(test_df)
    
    # Evaluate model
    # AUC-ROC
    auc_evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    auc = auc_evaluator.evaluate(rf_predictions)
    
    # Accuracy
    accuracy_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy"
    )
    accuracy = accuracy_evaluator.evaluate(rf_predictions)
    
    # Precision
    precision_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedPrecision"
    )
    precision = precision_evaluator.evaluate(rf_predictions)
    
    # Recall
    recall_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedRecall"
    )
    recall = recall_evaluator.evaluate(rf_predictions)
    
    # F1 Score
    f1_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="f1"
    )
    f1 = f1_evaluator.evaluate(rf_predictions)
    
    # Log metrics
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    # Log model
    mlflow.spark.log_model(rf_model, "random_forest_model")
    
    # Store run ID for later use
    rf_run_id = run.info.run_id
    
    print("\n📊 Random Forest Performance Metrics:")
    print(f"  AUC-ROC:   {auc:.4f}")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"\n✅ Model logged to MLflow (Run ID: {rf_run_id})")

In [ ]:
# Extract and visualize feature importance from Random Forest
rf_model_stage = rf_model.stages[-1]  # Get the Random Forest stage
feature_importances = rf_model_stage.featureImportances.toArray()

# Create DataFrame with feature names and importance scores
importance_df = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

# Display top 20 features
print("\n🔝 Top 20 Most Important Features:")
print(importance_df.head(20).to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(12, 8))
plt.barh(importance_df['Feature'].head(20), importance_df['Importance'].head(20))
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.title('Top 20 Feature Importances - Random Forest Model')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Save to MLflow
with mlflow.start_run(run_id=rf_run_id):
    mlflow.log_figure(plt.gcf(), "feature_importance.png")
    
print("✅ Feature importance visualization saved to MLflow")

### 5.2 Logistic Regression

In [ ]:
# Train Logistic Regression Model
with mlflow.start_run(run_name="Logistic_Regression_v1") as run:
    
    print("📈 Training Logistic Regression...")
    
    # Log parameters
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 100)
    mlflow.log_param("reg_param", 0.01)
    mlflow.log_param("elasticnet_param", 0.0)
    mlflow.log_param("train_test_split", "80/20")
    mlflow.log_param("random_seed", 42)
    mlflow.log_param("feature_count", len(feature_columns))
    
    # Define Logistic Regression model
    lr_classifier = LogisticRegression(
        labelCol="label",
        featuresCol="features",
        maxIter=100,
        regParam=0.01,
        elasticNetParam=0.0,  # L2 regularization
        family="binomial"
    )
    
    # Create pipeline
    lr_pipeline = Pipeline(stages=[assembler, scaler, lr_classifier])
    
    # Train model
    lr_model = lr_pipeline.fit(train_df)
    
    # Make predictions on test set
    lr_predictions = lr_model.transform(test_df)
    
    # Evaluate model
    auc = auc_evaluator.evaluate(lr_predictions)
    accuracy = accuracy_evaluator.evaluate(lr_predictions)
    precision = precision_evaluator.evaluate(lr_predictions)
    recall = recall_evaluator.evaluate(lr_predictions)
    f1 = f1_evaluator.evaluate(lr_predictions)
    
    # Log metrics
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    # Log model
    mlflow.spark.log_model(lr_model, "logistic_regression_model")
    
    # Store run ID
    lr_run_id = run.info.run_id
    
    print("\n📊 Logistic Regression Performance Metrics:")
    print(f"  AUC-ROC:   {auc:.4f}")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"\n✅ Model logged to MLflow (Run ID: {lr_run_id})")

### 5.3 Gradient Boosted Trees (GBT)

In [ ]:
# Train Gradient Boosted Trees Model
with mlflow.start_run(run_name="Gradient_Boosted_Trees_v1") as run:
    
    print("🚀 Training Gradient Boosted Trees...")
    
    # Log parameters
    mlflow.log_param("model_type", "GBTClassifier")
    mlflow.log_param("max_iter", 50)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("step_size", 0.1)
    mlflow.log_param("train_test_split", "80/20")
    mlflow.log_param("random_seed", 42)
    mlflow.log_param("feature_count", len(feature_columns))
    
    # Define GBT model
    gbt_classifier = GBTClassifier(
        labelCol="label",
        featuresCol="features",
        maxIter=50,
        maxDepth=5,
        stepSize=0.1,
        seed=42
    )
    
    # Create pipeline
    gbt_pipeline = Pipeline(stages=[assembler, scaler, gbt_classifier])
    
    # Train model
    gbt_model = gbt_pipeline.fit(train_df)
    
    # Make predictions on test set
    gbt_predictions = gbt_model.transform(test_df)
    
    # Evaluate model
    auc = auc_evaluator.evaluate(gbt_predictions)
    accuracy = accuracy_evaluator.evaluate(gbt_predictions)
    precision = precision_evaluator.evaluate(gbt_predictions)
    recall = recall_evaluator.evaluate(gbt_predictions)
    f1 = f1_evaluator.evaluate(gbt_predictions)
    
    # Log metrics
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    # Log model
    mlflow.spark.log_model(gbt_model, "gbt_model")
    
    # Store run ID
    gbt_run_id = run.info.run_id
    
    print("\n📊 Gradient Boosted Trees Performance Metrics:")
    print(f"  AUC-ROC:   {auc:.4f}")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"\n✅ Model logged to MLflow (Run ID: {gbt_run_id})")

## 6. Model Comparison & Selection

In [ ]:
# Compare all three models
print("\n🏆 Model Performance Comparison\n")
print("=" * 80)

# Evaluate Random Forest on test set
rf_test_predictions = rf_model.transform(test_df)
rf_metrics = {
    'Model': 'Random Forest',
    'AUC-ROC': auc_evaluator.evaluate(rf_test_predictions),
    'Accuracy': accuracy_evaluator.evaluate(rf_test_predictions),
    'Precision': precision_evaluator.evaluate(rf_test_predictions),
    'Recall': recall_evaluator.evaluate(rf_test_predictions),
    'F1 Score': f1_evaluator.evaluate(rf_test_predictions)
}

# Evaluate Logistic Regression on test set
lr_test_predictions = lr_model.transform(test_df)
lr_metrics = {
    'Model': 'Logistic Regression',
    'AUC-ROC': auc_evaluator.evaluate(lr_test_predictions),
    'Accuracy': accuracy_evaluator.evaluate(lr_test_predictions),
    'Precision': precision_evaluator.evaluate(lr_test_predictions),
    'Recall': recall_evaluator.evaluate(lr_test_predictions),
    'F1 Score': f1_evaluator.evaluate(lr_test_predictions)
}

# Evaluate GBT on test set
gbt_test_predictions = gbt_model.transform(test_df)
gbt_metrics = {
    'Model': 'Gradient Boosted Trees',
    'AUC-ROC': auc_evaluator.evaluate(gbt_test_predictions),
    'Accuracy': accuracy_evaluator.evaluate(gbt_test_predictions),
    'Precision': precision_evaluator.evaluate(gbt_test_predictions),
    'Recall': recall_evaluator.evaluate(gbt_test_predictions),
    'F1 Score': f1_evaluator.evaluate(gbt_test_predictions)
}

# Create comparison DataFrame
comparison_df = pd.DataFrame([rf_metrics, lr_metrics, gbt_metrics])
comparison_df = comparison_df.round(4)

print(comparison_df.to_string(index=False))
print("=" * 80)

# Determine best model based on AUC-ROC
best_model_row = comparison_df.loc[comparison_df['AUC-ROC'].idxmax()]
print(f"\n🥇 Best Model: {best_model_row['Model']} (AUC-ROC: {best_model_row['AUC-ROC']:.4f})")

## 7. Batch Scoring - Generate Predictions for All Employees

In [ ]:
# Use the best model (Random Forest) for batch scoring
print("🔮 Generating predictions for all employees using Random Forest model...\n")

# Score all employees
all_predictions = rf_model.transform(ml_data)

# Extract probability of attrition (probability of class 1)
# The probability column is a Vector, we need to extract the second element (probability of attrition)
predictions_final = all_predictions.select(
    F.col('EmpID'),
    F.col('is_active'),
    F.col('label').alias('actual_attrition'),
    F.col('prediction').cast('int').alias('predicted_attrition'),
    F.round(F.element_at(F.col('probability'), 2), 4).alias('attrition_probability'),
    F.when(F.element_at(F.col('probability'), 2) >= 0.7, 'Critical')
     .when(F.element_at(F.col('probability'), 2) >= 0.5, 'High')
     .when(F.element_at(F.col('probability'), 2) >= 0.3, 'Medium')
     .otherwise('Low').alias('risk_level'),
    F.current_timestamp().alias('prediction_timestamp')
)

print(f"✅ Predictions generated for {predictions_final.count():,} employees")

# Display sample predictions
print("\n📋 Sample Predictions (Top 20 highest risk):")
display(predictions_final.orderBy(F.col('attrition_probability').desc()).limit(20))

In [ ]:
# Analyze risk distribution
print("\n📊 Risk Level Distribution:\n")
risk_distribution = predictions_final.groupBy('risk_level').count().orderBy('count', ascending=False)
display(risk_distribution)

# Count high-risk employees (Critical + High)
high_risk_count = predictions_final.filter(
    F.col('risk_level').isin(['Critical', 'High'])
).count()

total_employees = predictions_final.count()
print(f"\n⚠️ High-Risk Employees (Critical + High): {high_risk_count:,} ({high_risk_count/total_employees*100:.2f}%)")

## 8. Save Predictions to Gold Layer

In [ ]:
# Save predictions to Delta table in Gold layer
print("💾 Saving predictions to workspace.gold.gold_attrition_predictions...\n")

predictions_final.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.gold_attrition_predictions")

print("✅ Predictions successfully saved to gold.gold_attrition_predictions")
print("\nTable location: workspace.gold.gold_attrition_predictions")
print(f"Records saved: {predictions_final.count():,}")
print("\nThis table is now ready for Power BI dashboards! 📊")

## 9. Model Registration to MLflow Model Registry

In [ ]:
# Register the best model (Random Forest) to Model Registry
model_name = "employee_attrition_predictor"

print(f"📦 Registering Random Forest model to MLflow Model Registry as '{model_name}'...\n")

# Register model
model_uri = f"runs:/{rf_run_id}/random_forest_model"

try:
    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )
    
    print(f"✅ Model registered successfully!")
    print(f"   Model Name: {registered_model.name}")
    print(f"   Model Version: {registered_model.version}")
    print(f"   Run ID: {rf_run_id}")
    
except Exception as e:
    print(f"⚠️ Model registration note: {str(e)}")
    print("   Model can still be loaded directly from the run ID.")

## 10. Summary & Next Steps

In [ ]:
print("\n" + "="*80)
print("🎉 ML MODEL TRAINING COMPLETE")
print("="*80)

print("\n✅ Completed Tasks:")
print("   1. Loaded 27 ML features from gold.gold_ml_features")
print("   2. Trained 3 models: Random Forest, Logistic Regression, GBT")
print("   3. Logged all experiments to MLflow with metrics and parameters")
print("   4. Generated predictions for all employees")
print("   5. Saved predictions to gold.gold_attrition_predictions table")
print("   6. Registered best model to MLflow Model Registry")

print("\n📊 Key Outputs:")
print(f"   • MLflow Experiment: /employee-attrition-prediction")
print(f"   • Best Model: Random Forest")
print(f"   • Predictions Table: workspace.gold.gold_attrition_predictions")
print(f"   • High-Risk Employees Identified: {high_risk_count:,}")

print("\n🚀 Next Steps:")
print("   1. Connect Power BI to gold.gold_attrition_predictions")
print("   2. Build Executive Dashboard with risk scores and KPIs")
print("   3. Create HR Manager Dashboard with at-risk employee lists")
print("   4. Build Predictive Analytics Dashboard with feature importance")
print("   5. Set up Databricks Workflows for automated retraining")

print("\n💡 Power BI Usage:")
print("   • Use 'attrition_probability' for gauge charts and risk meters")
print("   • Filter by 'risk_level' for segmented views (Critical/High/Medium/Low)")
print("   • Join with employee data for demographic drill-downs")
print("   • Create alerts for employees with risk_level = 'Critical'")

print("\n" + "="*80)
print(f"Execution completed at: {datetime.now()}")
print("="*80)